In [0]:
# Current schemas to archive
schemas_to_archive = {
    "bronze": "archive_pre_final_bronze",
    "silver": "archive_pre_final_silver",
    "gold": "archive_pre_final_gold",
    "mimic_iv_3_1_bronze": "archive_pre_final_mimic_iv_3_1_bronze"
}

# Create archive schemas
for source_schema, archive_schema in schemas_to_archive.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{archive_schema}")
    print(f"READY: workspace.{archive_schema}")

In [0]:
# One consolidated archive schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.archive")

# Existing schemas to consolidate
archive_sources = {
    "archive_mimic3_bronze": "mimic3_bronze",
    "archive_mimic3_silver": "mimic3_silver",

    "archive_pre_final_bronze": "prefinal_bronze",
    "archive_pre_final_silver": "prefinal_silver",
    "archive_pre_final_gold": "prefinal_gold",
    "archive_pre_final_mimic_iv_3_1_bronze": "prefinal_mimiciv31_bronze",

    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "mimic_iv_3_1_bronze": "mimiciv31_bronze"
}

moves = []

for source_schema, prefix in archive_sources.items():

    if spark.catalog.databaseExists(f"workspace.{source_schema}"):

        tables = spark.sql(
            f"SHOW TABLES IN workspace.{source_schema}"
        ).collect()

        for row in tables:
            source = f"workspace.{source_schema}.{row.tableName}"
            target = f"workspace.archive.{prefix}_{row.tableName}"

            moves.append((source, target))

print(f"Tables to archive: {len(moves)}\n")

for source, target in moves:
    print(f"{source}")
    print(f"  -> {target}\n")

In [0]:
# Move all tables into workspace.archive

moved = 0
skipped = 0
failed = []

for source, target in moves:
    try:
        # Safety check: do not overwrite an existing archive table
        if spark.catalog.tableExists(target):
            print(f"SKIP - target already exists: {target}")
            skipped += 1
            continue

        spark.sql(f"ALTER TABLE {source} RENAME TO {target}")
        print(f"MOVED: {source} -> {target}")
        moved += 1

    except Exception as e:
        print(f"FAILED: {source}")
        print(str(e))
        failed.append(source)

print()
print(f"Moved:   {moved}")
print(f"Skipped: {skipped}")
print(f"Failed:  {len(failed)}")


In [0]:
archive_tables = spark.sql(
    "SHOW TABLES IN workspace.archive"
).collect()

print(f"Tables now in workspace.archive: {len(archive_tables)}")

for row in archive_tables:
    print(row.tableName)

In [0]:
# Old schemas that should now be empty
schemas_to_remove = [
    "archive_mimic3_bronze",
    "archive_mimic3_silver",
    "archive_pre_final_bronze",
    "archive_pre_final_silver",
    "archive_pre_final_gold",
    "archive_pre_final_mimic_iv_3_1_bronze",
    "bronze",
    "silver",
    "gold",
    "mimic_iv_3_1_bronze"
]

for schema in schemas_to_remove:
    full_schema = f"workspace.{schema}"

    try:
        remaining = spark.sql(
            f"SHOW TABLES IN {full_schema}"
        ).collect()

        if len(remaining) == 0:
            spark.sql(f"DROP SCHEMA IF EXISTS {full_schema}")
            print(f"REMOVED: {full_schema}")
        else:
            print(f"KEPT: {full_schema} still contains {len(remaining)} objects")

    except Exception as e:
        print(f"CHECK: {full_schema} -> {str(e)}")